# Create Wenner-Gren Foundation Awards (GRANT/FELLOWSHIP PATTERN, method-5 static HTML)

Wenner-Gren Foundation grantees from the foundation's own WordPress-published grantee archive at wennergren.org. Major US-based anthropology funder (Hunt Postdoctoral Fellowships, Wadsworth International Fellowships, Engaged Anthropology Grants, Post-PhD Research Grants, etc.).

**Prerequisites:**
- Run `scripts/local/wenner_gren_to_s3.py` first.

**Data source:** 3 sitemap files (`/grantee-sitemap.xml`, `-sitemap2.xml`, `-sitemap3.xml`) enumerate **2,161 grantee URLs**. Per grantee, `wennergren.org/grantee/{slug}/` is a server-rendered page with a clean `<div class="c-grantee-block">` block structure exposing Grant Type, Institutional Affiliation, Grant number, Approve Date, Project Title.

**S3 location:** `s3a://openalex-ingest/awards/wenner_gren/wenner_gren_grantees.parquet`

**Awarding body:**
- funder_id: 4320306550
- display_name: "Wenner-Gren Foundation"
- country: US
- ROR: none
- DOI: 10.13039/100001388

**Coverage (from 2026-05-22 local smoke run on first 10 grantees — full run pending):**
- 100% grantee_name / slug / grant_type / affiliation / grant_number / approve_year / project_title
- Native award ID = `grant_number` (e.g., "Gr. WIF-294"). `funder_award_id = wg-{slug-of-grant-number}-{slug}` (slug-disambiguated in case of multi-year grant_number reuse).

**Programs (visible in `grant_type` field):**
- Hunt Postdoctoral Fellowship
- Wadsworth International Fellowship (`Gr. WIF-*`)
- Engaged Anthropology Grant
- Post-PhD Research Grant
- Dissertation Fieldwork Grant
- Conference and Workshop Grant
- Historical Archives Program (older grants)

**Amount/currency:** NULL with §6.7 waiver. Wenner-Gren publishes program-level fixed amounts on its /funding-opportunities/ pages (Hunt Postdoc: $42k, Wadsworth: $25k/yr × 2y, Engaged Anthropology: up to $5k) but does NOT publish per-grantee amounts on the grantee archive. Grant/fellowship-pattern precedent for NULL amount: HHMI #44, CIFAR #79, Damon Runyon #73, Packard #95, Rita Allen #107, Schmidt Sciences #108, NOMIS #109.

**Provenance:** `wenner_gren_grantees` (direct from foundation, not an aggregator).

**Priority:** 110 (UCOP 106 / Rita Allen 107 / Schmidt Sciences 108 / NOMIS 109 are immediately-prior slots in flight; Vilcek at 105 is current main registry head).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wenner_gren_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/wenner_gren/wenner_gren_grantees.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.wenner_gren_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.wenner_gren_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

Wenner-Gren is non-monetary in our schema (§6.7 waiver). Money-field scan is run for completeness; we expect zero matches.

In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.wenner_gren_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.wenner_gren_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT
    grantee_name, slug, grant_type, affiliation, grant_number,
    approve_date_raw, approve_year,
    SUBSTR(project_title, 1, 150) AS title_preview
FROM openalex.awards.wenner_gren_raw LIMIT 5;


In [ ]:
%sql
-- Native key uniqueness (grant_number is the canonical award ID)
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT grant_number) AS distinct_grant_numbers,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.wenner_gren_raw;


In [ ]:
%sql
-- Year distribution (Wenner-Gren has been funding since the 1940s — expect a long historical tail)
SELECT approve_year, COUNT(*) as grantees
FROM openalex.awards.wenner_gren_raw
WHERE approve_year IS NOT NULL
GROUP BY approve_year ORDER BY approve_year DESC LIMIT 30;


In [ ]:
%sql
-- Grant program (grant_type) distribution
SELECT grant_type, COUNT(*) as grantees
FROM openalex.awards.wenner_gren_raw
GROUP BY grant_type
ORDER BY grantees DESC LIMIT 20;


In [ ]:
%sql
-- Top affiliations
SELECT affiliation, COUNT(*) as grantees
FROM openalex.awards.wenner_gren_raw
GROUP BY affiliation
ORDER BY grantees DESC LIMIT 20;


In [ ]:
%sql
-- Approve-date coverage by decade
SELECT
    CONCAT(CAST(FLOOR(CAST(approve_year AS INT) / 10) * 10 AS INT), 's') AS decade,
    COUNT(*) AS total,
    COUNT(approve_date_iso) AS with_iso_date,
    ROUND(COUNT(approve_date_iso) * 100.0 / COUNT(*), 1) AS pct_iso_date
FROM openalex.awards.wenner_gren_raw
WHERE approve_year IS NOT NULL
GROUP BY FLOOR(CAST(approve_year AS INT) / 10)
ORDER BY decade DESC;


## Step 1.6: Fail-fast — Verify the Wenner-Gren Funder Row Exists

**Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320306550;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wenner_gren_awards
USING delta
AS
WITH
wg_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306550
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(approve_year AS INT) AS parsed_year,
        TRY_TO_DATE(approve_date_iso, 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.wenner_gren_raw
    WHERE grantee_name IS NOT NULL
      AND TRIM(grantee_name) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        -- display_name = grantee name + project hint when short
        g.grantee_name as display_name,

        g.project_title as description,

        f.funder_id,
        g.funder_award_id,

        CAST(NULL AS DOUBLE) as amount,    -- §6.7 waiver
        CAST(NULL AS STRING) as currency,  -- §6.7 waiver

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- funding_type: 'fellowship' for Fellowship-named grant types, else 'research'
        CASE
            WHEN LOWER(COALESCE(g.grant_type, '')) RLIKE 'fellowship|scholarship|postdoc|dissertation|traineeship' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.grant_type, '')) RLIKE 'conference|workshop|symposium' THEN 'travel'
            ELSE 'research'
        END as funding_type,

        -- funder_scheme = the grant_type when populated, else "Wenner-Gren Foundation Grant"
        COALESCE(NULLIF(TRIM(g.grant_type), ''), 'Wenner-Gren Foundation Grant') as funder_scheme,

        'wenner_gren_grantees' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        -- lead_investigator = the grantee themselves (fellowship/research-grant pattern)
        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.affiliation), '') as name,
                CAST(NULL AS STRING) as country,  -- affiliation includes country prefix sometimes; not split out
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN wg_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 110

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wenner_gren_grantees' AND priority = 110;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    110 as priority  -- Wenner-Gren priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.wenner_gren_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.wenner_gren_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.wenner_gren_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(landing_page_url) as has_url,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) as pct_description,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) as pct_pi
FROM openalex.awards.wenner_gren_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, funder_scheme, funding_type,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name AS pi_affiliation,
       landing_page_url
FROM openalex.awards.wenner_gren_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.wenner_gren_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.wenner_gren_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 30;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.wenner_gren_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.wenner_gren_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;


In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt
FROM openalex.awards.wenner_gren_awards
GROUP BY funding_type ORDER BY cnt DESC;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.wenner_gren_awards;
